In [12]:
###-----------------------------
# Michael Hawthorne (Term Project Main Notebook for GIS-5653)
# Title:
# Identifying Public Schools Near Flood-Hazard Areas in the Texas Golden Triangle
###-----------------------------

# Imports and Setup:
import arcpy
arcpy.env.overwriteOutput = True # used to overwrite existing file names

# import module
import term_project_module

#-------------------------------

# set workspace and file paths for selected layers and output locations:
# Update these paths to match the location of this data on your local computer.
project_data = r"path/to/project/data"
output_gdb = r"path/to/output/output.gdb"

county_layer = project_data + r"FinalProject.gdb/County_Boundaries"
# path for the selected county polygon output under the name Selected_County
selected_county = output_gdb + r"/Selected_County"
# path for the jefferson county flood zones layer
jefferson_fldzone = r"/FinalProject.gdb/Jefferson_County_Flood"
# path for the state flood zones layer
state_flood = r"/FinalProject.gdb/State_Flood_Zones"
# path for the school point layer
school_layer = r"Schools_2024_to_2025/Schools_2024_to_2025.shp"

#-------------------------
# county input:
county_choice = term_project_module.input_county_choice() # runs the county selection and user input through the module
term_project_module.county_selection(county_layer, county_choice, selected_county)

print(f"{county_choice} County has been chosen as the selection.", # prints the valid county selection name from the module
     "Loading next selection, please wait.") # asks the user to wait for the next selection (takes a while to load)

# county projection:
projected_county = output_gdb + r"/Selected_County_UTM" # sets the output path for the projected county output
# uses projection layer module to project the user selected county
term_project_module.projection_layer(selected_county, projected_county)

#-------------------------
# flood zone choice: # If Jefferson county is chosen, the jefferson_fldzone will be used, if not, then state_flood will be used
selected_flood_zone = term_project_module.flood_zone(county_choice, jefferson_fldzone, state_flood)
# creates the new county flood zone output and sends it to the output gdb under the name County_Flood_Zone
county_flood_selection = output_gdb + r"/County_Flood_Zone"
# call the function flood zone clip function
county_flood = term_project_module.flood_zone_clip(county_choice, selected_flood_zone, projected_county, county_flood_selection)

# projects the county flood areas to a proper spatial reference for Southeast Texas:
projected_flood = output_gdb + r"/County_Flood_Zones_UTM"
# uses the projection layer module to project the selected county flood areas
term_project_module.projection_layer(county_flood, projected_flood)

# call special flood hazard area selection module and sets the output name to Selected_SFHA:
selected_sfha = output_gdb + r"/Selected_SFHA"
# calls the special flood hazard selection module
term_project_module.sfha_selection(projected_flood, selected_sfha)

# calls the dissolve flood hazard area function from module and sets the output name to Dissolved_SFHA:
dissolved_sfha = output_gdb + r"/Dissolved_SFHA"
# dissolves all SFHA polygons that passed the initial selection using the dissolve flood zone module
term_project_module.dissolve_flood_zone(selected_sfha, dissolved_sfha)

#-------------------------
# buffer zone choice:
# asks for user input for buffer zone distance from buffer module:
buffer_distance = term_project_module.input_buffer_distance()

# creates the buffer output path and names the output Buffered_SFHA:
buffered_sfha = output_gdb + r"/Buffered_SFHA"

# will either return the original flood zone or create the valid buffer using the user defined distance:
flood_analysis_area = term_project_module.flood_buffer(dissolved_sfha, buffer_distance, buffered_sfha)

#-------------------------
# results output:
# creates the output path for the schools selection and names the output Selected_Schools:
selected_schools = output_gdb + r"/Selected_Schools"

# calls the school selection function from module
term_project_module.school_selection(school_layer, county_choice, selected_schools)

# projects selected schools to a proper projection for Southeast Texas
projected_schools = output_gdb + r"/Selected_Schools_UTM"
# calls the projection layer module 
term_project_module.projection_layer(selected_schools, projected_schools)

# creates the at-risk schools selection output path and names output At_Risk_Schools:
at_risk_school_output = output_gdb + r"/At_Risk_Schools"

# calls the at-risk schools function from module:
at_risk_schools = term_project_module.at_risk_schools(projected_schools, flood_analysis_area, at_risk_school_output)

# calls the school count module for the count of schools selected in analysis
number_at_risk = term_project_module.school_count(at_risk_schools)

# prints the number of at-risk schools selected through analysis:
print(f"{number_at_risk} is the number of schools that are within the specified flood hazard buffer area.")
print("All created layers have been saved to the specified gdb Output path location.")

# uses a Search Cursor to browse and then print the specified school names that are within the analysis:
with arcpy.da.SearchCursor(at_risk_school_output, ["USER_Sch_1"]) as cursor: # uses a cursor to search for School Name
    print("Here is a list of all of the schools in the specified criteria: ") # prints a statement for the specified header
    print("School Name:") # added a School Name heading for the printout
    print("------------") # adds an additional space to separate header from the output data
    for row in cursor: # uses a 'for' loop to print USER_Sch_1 field for the selected schools
        print(row[0])

Please type either 'Jefferson', 'Hardin', or 'Orange' to choose your county:  Orange


Orange County has been chosen as the selection. Loading next selection, please wait.


Enter a buffer distance number (in miles) from the flood zone: Entering a value of '0' will include all schools that are directly in a flood zone:  .15


Please wait on your selection output.
16 is the number of schools that are within the specified flood hazard buffer area.
All created layers have been saved to the specified gdb Output path location.
Here is a list of all of the schools in the specified criteria: 
School Name:
------------
TEKOA ACADEMY OF ACCELERATED STUDIES - ORANGE
BRIDGE CITY H S
BRIDGE CITY MIDDLE
BRIDGE CITY EL
BRIDGE CITY INT
ORANGEFIELD H S
WEST ORANGE-STARK H S
ORANGEFIELD J H
ORANGEFIELD EL
M B NORTH E C LRN CTR
WEST ORANGE-STARK MIDDLE
PINE FOREST EL
OAK FOREST EL
VIDOR MIDDLE
LITTLE CYPRESS J H
LITTLE CYPRESS EL
